# PersonaPlex on Colab — voice cloning + Q&A in the cloned voice

Runs the `personaplex` model from the fork **`un1876/transformers`**: converts the gated
**`nvidia/personaplex-7b-v1`** checkpoint (keeping all 16 Depformer slices), clones a
reference voice, and answers your recorded question in that voice.

**Runtime requirements** (`Runtime > Change runtime type`):
- Conversion (step 4): **High-RAM** (~30 GB peak) — Colab Pro. No GPU needed for this step.
- Inference (step 6+): GPU with **≥16 GB VRAM** (L4 / A100 recommended).

**Before starting**: accept the license at
<https://huggingface.co/nvidia/personaplex-7b-v1> — with the same HF account you log in with below.

> Note: this is **offline** generation (one question → one answer). Real-time full-duplex
> conversation needs the kyutai serving stack, not `transformers`.

In [ ]:
# 0) Environment check
import subprocess


print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no GPU (fine for conversion, not for inference)")
meminfo = subprocess.check_output(["bash", "-c", "awk '/MemTotal/ {print $2}' /proc/meminfo"])
print("CPU RAM:", round(int(meminfo) / 1024**2, 1), "GB  (conversion needs ~30 GB → High-RAM runtime)")

## 1) Install `transformers` from the fork

Colab pre-installs an older `transformers` that is **already loaded in this kernel**.
Uninstall it first, install the fork, then **restart the runtime** — without the restart,
`import transformers` keeps returning the cached old version and
`transformers.models.personaplex` will raise `ModuleNotFoundError`.

In [ ]:
# 1) ~2-3 min
!pip uninstall -y transformers
!pip install -q "git+https://github.com/un1876/transformers.git@main" soundfile

### ⚠️ Now: `Runtime > Restart session`

The install survives the restart (it is on disk). After restarting, continue from the next cell —
do **not** re-run the install.

In [ ]:
# 2) Run AFTER the restart — verifies the fork is active.
import transformers
import transformers.models.personaplex as personaplex_module


print("transformers", transformers.__version__)  # expect 5.14.0.dev0 (fork)
print("personaplex at", personaplex_module.__path__[0])

## 2) Authenticate + verify gated access (before the 17 GB download)

Log in, confirm **which account** the token belongs to, and probe the gated repo with a
few-KB metadata request. If this cell fails, fix access first — the big download would
fail with the same `403 GatedRepoError` anyway.

In [ ]:
# 3) Login + access probe
import os

from huggingface_hub import get_safetensors_metadata, login, whoami


token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata

    token = token or userdata.get("HF_TOKEN")
except Exception as exc:  # secret not set / not on Colab
    print("Colab secret HF_TOKEN not available:", exc)
login(token=token)  # interactive prompt if token is None

print("logged in as:", whoami()["name"], "← the license must be accepted by THIS account")
try:
    meta = get_safetensors_metadata("nvidia/personaplex-7b-v1")
except Exception as exc:
    raise RuntimeError(
        "No gated access yet. Visit https://huggingface.co/nvidia/personaplex-7b-v1 with the "
        "account above, request access / accept the license, wait for approval, then re-run."
    ) from exc
names = list(meta.weight_map)
lin_slices = sorted({n.split(".")[1] for n in names if n.startswith("linears.")}, key=int)
print(len(names), "tensors | Depformer lm-head slices:", lin_slices)  # expect 0..15 (dep_q=16)

## 3) Convert the checkpoint → HF `Personaplex` format

Downloads ~17 GB (PersonaPlex) + ~15.6 GB (moshiko base, for the Mimi graft), then converts.
Progress prints `[1/6] … [6/6]` — the silent stretch after `[3/6]` is normal CPU work
(a few minutes); it is done only after `[6/6] … done` appears.

Already converted? The cell skips itself. Point `OUT_DIR` at Google Drive to persist it.

*(No `--download-voices`: the repo's official voice prompts are precomputed `.pt`
embeddings for the kyutai stack, not audio files — we upload a reference wav instead.)*

In [ ]:
# 4) Convert (skips if OUT_DIR already contains a converted model)
import os


OUT_DIR = "/content/personaplex-hf"  # e.g. "/content/drive/MyDrive/personaplex-hf" to persist

if os.path.exists(f"{OUT_DIR}/config.json"):
    print("already converted — skipping")
else:
    !python -m transformers.models.personaplex.convert_personaplex_transformers --out {OUT_DIR}

## 4) Load the converted model

`personaplex` is not registered in the `Auto*` mappings — import the class directly.

In [ ]:
# 5) Load (needs the GPU runtime from here on)
import torch

from transformers import AutoTokenizer
from transformers.models.personaplex.modeling_personaplex import PersonaplexForConditionalGeneration


OUT_DIR = "/content/personaplex-hf"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = PersonaplexForConditionalGeneration.from_pretrained(OUT_DIR, dtype=torch.bfloat16).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(OUT_DIR)
FRAME_SIZE = int(model.config.sampling_rate / model.config.audio_encoder_config.frame_rate)  # 1920 @ 24kHz
print("loaded on", device, "| depth dep_q =", model.config.depth_decoder_config.num_codebooks,
      "| frame size =", FRAME_SIZE)

## 5) Upload a reference voice

Any clean 5–10 s speech clip works (your own voice, or e.g.
`_personaplex_ref/assets/test/input_assistant.wav` from the reference repo — 24 kHz mono PCM).
Other sample rates are resampled automatically; −24 LUFS loudness is recommended but optional.

In [ ]:
# 6) Voice upload (also defines the wav loader used by later cells)
import soundfile as sf
import torch
import torchaudio
from google.colab import files


def normalize_loudness(wav, target_lufs=-24.0):
    # PersonaPlex preprocessing spec is -24 LUFS; RMS approximation (BS.1770 -0.691 offset).
    # Un-normalized recordings (e.g. phone mics) push Mimi codes out of distribution and
    # the continuation can degrade into noise.
    rms = wav.pow(2).mean().sqrt().clamp_min(1e-8)
    gain_db = target_lufs - (20.0 * torch.log10(rms).item() - 0.691)
    wav = wav * (10.0 ** (gain_db / 20.0))
    peak = wav.abs().max()
    if peak > 0.99:
        wav = wav * (0.99 / peak)
    return wav


def load_wav_24k_mono(path):
    wav, sr = sf.read(path, always_2d=True)  # (T, C)
    wav = torch.tensor(wav.mean(axis=1), dtype=torch.float32)  # mono
    if sr != 24000:
        wav = torchaudio.functional.resample(wav, sr, 24000)
    return normalize_loudness(wav)


uploaded = files.upload()  # choose the reference .wav
assert uploaded, "no file uploaded — run the cell again and pick a wav file"
voice = load_wav_24k_mono(next(iter(uploaded)))
print("reference voice:", round(voice.numel() / 24000, 1), "s,", voice.numel() // FRAME_SIZE, "frames")

## 6) Voice cloning — persona monologue

Builds the `voice → silence → role text → silence` prefill and generates ~10 s in the cloned
voice. The raw output waveform **replays the prefill** (voice prompt + silences) before the new
speech, so we trim `prefill_frames × FRAME_SIZE` samples.

In [ ]:
# 7) Cloned-voice monologue
import torch
from IPython.display import Audio, display

from transformers import StoppingCriteria, StoppingCriteriaList


class StopOnSilence(StoppingCriteria):
    # Stops generation once the model has spoken and then stays silent (text PAD=3)
    # for `silence_sec` seconds - i.e. ends at the sentence instead of a fixed length.
    def __init__(self, prefill_len, silence_sec=1.0, pad_id=3):
        self.prefill, self.pad = prefill_len, pad_id
        self.n = int(silence_sec * 12.5)

    def __call__(self, input_ids, scores, **kwargs):
        gen = input_ids[0, self.prefill :]
        if (gen != self.pad).sum() == 0:  # has not started speaking yet
            return False
        return len(gen) >= self.n and bool((gen[-self.n :] == self.pad).all())


PERSONA_TEXT = "You are a calm, warm guide."

persona_ids = tokenizer(PERSONA_TEXT, add_special_tokens=False, return_tensors="pt").input_ids.to(device)
persona = model.build_persona_prompt(voice_input_values=voice.to(device), persona_input_ids=persona_ids)
prefill_frames = persona.input_ids.shape[-1]
print("prefill:", prefill_frames, "frames")

with torch.no_grad():
    out = model.generate(
        **persona,
        max_new_tokens=250,  # upper bound (~20 s); StopOnSilence ends it at the sentence
        stopping_criteria=StoppingCriteriaList([StopOnSilence(prefill_frames)]),
    )

# The output waveform replays the prefill first, and the delay mask trims a couple of
# leading frames — so cut the NEW frames from the END (robust) instead of skipping
# prefill samples from the start.
new_frames = out.sequences.shape[-1] - prefill_frames
full = out.audio_sequences[0, 0].float().cpu().numpy()
speech = full[-new_frames * FRAME_SIZE :]
if len(speech) > 24000:
    speech = speech[: -int(0.8 * 24000)]  # drop the trailing stopping-silence
sf.write("cloned.wav", speech, 24000)
print("generated", new_frames, "frames | saved cloned.wav:", round(len(speech) / 24000, 1), "s")
display(Audio(speech, rate=24000))

## 7) Ask a question — answered in the cloned voice

Record/prepare your question as a wav (24 kHz mono preferred), upload it, and the model answers
**in the cloned voice with the persona above**. The question rides on the *user* audio stream;
the agent stream stays silent during the prefill and then produces the reply.

This is offline single-turn: one uploaded question → one answer (not real-time full-duplex).

In [ ]:
# 8) Q&A in the cloned voice (requires `persona` from the previous cell)
import soundfile as sf
import torch
from google.colab import files
from IPython.display import Audio, display


uploaded = files.upload()  # choose your question .wav
assert uploaded, "no file uploaded — run the cell again and pick a wav file"
question = load_wav_24k_mono(next(iter(uploaded)))
print("question:", round(question.numel() / 24000, 1), "s")

# --- dialogue prefill assembled inline (build_dialogue_prompt was removed from the fork) ---
SILENCE = torch.tensor([948, 243, 1178, 546, 1736, 1030, 1978, 2008],
                       dtype=torch.long, device=device).view(1, -1, 1)
q = question.to(device=device, dtype=model.dtype).view(1, 1, -1)
r = q.shape[-1] % FRAME_SIZE
if r:
    q = torch.nn.functional.pad(q, (0, FRAME_SIZE - r))
with torch.no_grad():
    q_codes = model.audio_encoder.encode(q, num_quantizers=model.num_codebooks)[0]
T = q_codes.shape[2]

input_ids = torch.cat([persona.input_ids, torch.full((1, T), 3, dtype=torch.long, device=device)], dim=1)
user_codes = torch.cat([persona.user_audio_codes, q_codes], dim=2)                          # user stream: question
agent_codes = torch.cat([persona.personaplex_audio_codes, SILENCE.repeat(1, 1, T)], dim=2)  # agent stream: silence
inputs = dict(input_ids=input_ids, user_audio_codes=user_codes,
              personaplex_audio_codes=agent_codes, attention_mask=torch.ones_like(input_ids))
prefill_frames = input_ids.shape[-1]

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=250,  # upper bound; ends at the sentence via StopOnSilence
        stopping_criteria=StoppingCriteriaList([StopOnSilence(prefill_frames)]),
    )

new_frames = out.sequences.shape[-1] - prefill_frames  # actual generated frames
full = out.audio_sequences[0, 0].float().cpu().numpy()
answer = full[-new_frames * FRAME_SIZE :]  # robust end-based trim (see cell above)
if len(answer) > 24000:
    answer = answer[: -int(0.8 * 24000)]  # drop the trailing stopping-silence
sf.write("answer.wav", answer, 24000)
print("generated", new_frames, "frames (cap 250) | saved answer.wav:",
      round(len(answer) / 24000, 1), "s")
display(Audio(answer, rate=24000))

answer_text = tokenizer.decode(out.sequences[0, prefill_frames:], skip_special_tokens=True)
print("inner-monologue text:", answer_text[:300])
print("(문제 진단: 답변이 이상하면 ① 위 텍스트가 비었/깨졌는지 ② full 전체를 재생해 프리필 뒤에"
      " 정상 음성이 있는지 확인 — display(Audio(full, rate=24000)))")

### Follow-ups / troubleshooting
- Different persona: change `PERSONA_TEXT` and re-run cell 7 (same voice) or upload a new voice in cell 6.
- Longer answers: raise `max_new_tokens` (12.5 tokens ≈ 1 s).
- **Static / noise instead of speech**: usually an out-of-distribution input recording — the loader
  normalizes loudness to ≈−24 LUFS automatically, but very noisy / clipped recordings can still fail.
  Try a clean, close-mic 5–10 s clip. Also check the printed inner-monologue text: if it is empty or
  gibberish, the model did not engage with the question (bad question audio); if the text looks fine
  but audio is noisy, listen to the untrimmed `full` array to inspect where speech actually starts.